In [1]:
!pip install Pyro5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 2.5 MB/s eta 0:00:00


In [2]:
import Pyro5.api
import Pyro5.nameserver
import threading
import time

# ---------------- START NAME SERVER ----------------
def start_ns():
    Pyro5.nameserver.start_ns_loop(host="localhost")

ns_thread = threading.Thread(target=start_ns, daemon=True)
ns_thread.start()

time.sleep(2)  # Give time to start

# ---------------- REMOTE OBJECT ----------------
@Pyro5.api.expose
class StringService:
    def concat(self, a, b):
        return a + b

# ---------------- SERVER ----------------
def start_server():
    daemon = Pyro5.server.Daemon()

    # Locate name server
    ns = Pyro5.api.locate_ns()

    # Register object
    uri = daemon.register(StringService)

    # Register name → NO need to copy URI manually
    ns.register("string.concat", uri)

    print("Server ready. Registered as: string.concat")

    daemon.requestLoop()

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

Not starting broadcast server for IPv6.
NS running on localhost:9090 (::1)
URI = PYRO:Pyro.NameServer@localhost:9090


In [3]:
import Pyro5.api
import time

time.sleep(2)  # Ensure server is ready

# Connect using name (NO URI needed)
obj = Pyro5.api.Proxy("PYRONAME:string.concat")

# Take input
a = input("Enter first string: ")
b = input("Enter second string: ")

# Remote call
result = obj.concat(a, b)

print("Concatenated String:", result)

Server ready. Registered as: string.concat
Enter first string: sakshi
Enter second string: ghanwat
Concatenated String: sakshighanwat
